# GRPO 试跑（pilot）：Qwen2.5-3B-Instruct + QLoRA on MedQA

目的：不是看效果，而是测 **显存、每步耗时、训练是否稳定**，然后估算正式训练要多少 GPU 小时。

按顺序运行：检查 GPU → 设置变量 → 拉取 repo → 安装依赖 → 挂载 Drive →（可选）HF_TOKEN → 单元测试 → 数据样例 → 试跑 → 报告和图 → GPU 小时估算 → 复制结果到 Drive

运行前：菜单 **运行时 → 更改运行时类型 → GPU**（T4 或 A100）。

In [ ]:
# 1. 检查 GPU 和 Python 版本
!nvidia-smi
import sys, torch
print('Python', sys.version.split()[0])
print('torch', torch.__version__, '| CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| compute {major}.{minor} →', 'bf16' if major >= 8 else 'fp16')
assert torch.cuda.is_available(), '没有 GPU：运行时 → 更改运行时类型 → GPU'
assert sys.version_info >= (3, 11), 'TRL 1.16 需要 Python >= 3.11'

In [ ]:
# 2. 设置变量（只需要改这里）
REPO_URL = 'https://github.com/skjdsk/medqa-calibration.git'  # ← 改成你们的 repo
BRANCH   = 'main'
REPO_DIR = '/content/medqa-calibration'

OUTPUT_DIR = '/content/outputs/grpo_pilot_bf16_vllm'                    # Colab 本地输出目录（断开会丢）；文件夹名会出现在 Drive 的结果目录名里
DRIVE_DIR  = '/content/drive/MyDrive/medqa-calibration/grpo_pilot'      # Google Drive 保存目录

TRL_VERSION = '1.16.0'   # 代码按这个版本写；改版本前先看说明里的"需要检查的参数"

# 试跑参数（默认值与 train/grpo/grpo.py 一致）
N_QUESTIONS           = 64
SEED                  = 42
MAX_STEPS             = 30
NUM_GENERATIONS       = 4
MAX_COMPLETION_LENGTH = 512
LEARNING_RATE         = 1e-5
BETA                  = 0.04   # KL 系数；0 = 不算 KL（更快）
TEMPERATURE           = 1.0
PER_DEVICE_BS         = 4      # 每次前向/反向几条回答（爆显存先把它调小，同时调大 GRAD_ACCUM）
GRAD_ACCUM            = 4      # PER_DEVICE_BS × GRAD_ACCUM 必须是 NUM_GENERATIONS 的倍数
LORA_R, LORA_ALPHA    = 16, 32
LORA_TARGET_MODULES   = 'all-linear'
USE_VLLM              = True   # 只建议在 A100 上打开
QUANTIZATION          = 'none'  # '4bit' = QLoRA；'none' = bf16 + LoRA（开 vLLM 时用 'none'：QLoRA + vLLM 会让梯度逐步消失）

assert (PER_DEVICE_BS * GRAD_ACCUM) % NUM_GENERATIONS == 0, 'PER_DEVICE_BS × GRAD_ACCUM 必须能被 NUM_GENERATIONS 整除'
print('每步', PER_DEVICE_BS * GRAD_ACCUM, '条回答 =', PER_DEVICE_BS * GRAD_ACCUM // NUM_GENERATIONS, '题 ×', NUM_GENERATIONS, '个回答')

In [ ]:
# 3. git clone / pull repo
# 私有 repo：在 Colab 左侧 🔑（Secrets）里添加 GITHUB_TOKEN（GitHub 个人访问令牌），并打开"笔记本访问"
import os, subprocess
from pathlib import Path

def sh(cmd, shown=None):
    print('$', shown or cmd)
    subprocess.run(cmd, shell=True, check=True)

clone_url = REPO_URL
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://{gh_token}@')
except Exception:
    pass  # 没设置 GITHUB_TOKEN 就按公开 repo 处理

if (Path(REPO_DIR) / '.git').exists():
    sh(f'git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {clone_url} {REPO_DIR}', shown=f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sh('git log -1 --oneline')
for f in ['train/grpo/grpo_data.py', 'train/grpo/rewards.py', 'train/grpo/test_rewards.py', 'train/grpo/grpo.py', 'train/grpo/estimate.py']:
    assert Path(f).exists(), f'repo 里找不到 {f}：确认文件已经 push 到 {BRANCH}'
print('cwd =', os.getcwd())

In [ ]:
# 4. 安装依赖
from pathlib import Path
ip = get_ipython()
if Path('requirements.txt').exists():
    ip.run_line_magic('pip', 'install -q -r requirements.txt')
ip.run_line_magic('pip', f'install -q "trl[peft,quantization]=={TRL_VERSION}" "matplotlib" "pandas" "pytest"')
if USE_VLLM:
    # vLLM 会安装它自己需要的 torch 版本，装完后通常要 运行时 → 重启会话，再从"设置变量"那格重新运行
    # vLLM 0.28+ 删掉了 bitsandbytes 量化，而 TRL 在 QLoRA（4-bit）时会让 vLLM 用 bitsandbytes 加载模型
    # → 固定 0.27.1（最后一个支持 bitsandbytes 的版本，且在 TRL 1.16 允许的 0.22–0.31 范围内，要求 torch==2.13.0）
    ip.run_line_magic('pip', f'install -q "trl[vllm]=={TRL_VERSION}" "vllm==0.27.1"')

# Colab 预装的 torchao 0.10 太旧：peft 0.21 在不量化（--quantization none）加 LoRA 时会直接报错
# "only versions above 0.16.0 are supported"。我们用不到 torchao（vllm / peft 都不依赖它），直接卸载
ip.system('pip uninstall -y -q torchao')

import importlib.metadata as md
for pkg in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets'] + (['vllm'] if USE_VLLM else []):
    try:
        print(f'{pkg:14s}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:14s} 未安装')
print('\n如果上面 pip 提示需要重启（或之后 import 报错）：运行时 → 重启会话，然后从"设置变量"那格重新运行。')

In [ ]:
# 5. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive 保存目录：', DRIVE_DIR)

In [ ]:
# 6.（可选）设置 HF_TOKEN：Qwen2.5 和 MedQA 都是公开的，不设也能下载；设置后下载更稳定、不容易被限速
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('已设置 HF_TOKEN')
except Exception:
    print('没有找到 HF_TOKEN（可选），继续匿名下载')

In [ ]:
# 7. 跑打分函数的单元测试（几秒钟，不用 GPU）
!python -m pytest -q train/grpo/test_rewards.py

In [ ]:
# 8. 看数据样例：一条原始样例 + 转换后的 prompt（核对字段、核对 prompt 是否和 eval/ 一致）
!python -m train.grpo.grpo_data --n {N_QUESTIONS}

In [ ]:
# 9. 试跑（T4 约 30–60 分钟，A100 约 10–20 分钟；结束时自动把结果复制到 DRIVE_DIR）
args = [
    f'--n_questions {N_QUESTIONS}', f'--seed {SEED}', f'--max_steps {MAX_STEPS}',
    f'--num_generations {NUM_GENERATIONS}', f'--max_completion_length {MAX_COMPLETION_LENGTH}',
    f'--learning_rate {LEARNING_RATE}', f'--beta {BETA}', f'--temperature {TEMPERATURE}',
    f'--per_device_train_batch_size {PER_DEVICE_BS}', f'--gradient_accumulation_steps {GRAD_ACCUM}',
    f'--lora_r {LORA_R}', f'--lora_alpha {LORA_ALPHA}', f'--lora_target_modules {LORA_TARGET_MODULES}',
    f'--output_dir {OUTPUT_DIR}', f'--drive_dir {DRIVE_DIR}',
]
if USE_VLLM:
    args.append('--use_vllm')
args.append(f'--quantization {QUANTIZATION}')
ARGS = ' '.join(args)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('python -m train.grpo.grpo', ARGS)
!python -u -m train.grpo.grpo {ARGS} 2>&1 | tee {OUTPUT_DIR}/train_log.txt

In [ ]:
# 10. 看报告和图
import json
import pandas as pd
from IPython.display import Image, display

report = json.load(open(f'{OUTPUT_DIR}/pilot_report.json'))
print('GPU:', report['gpu_name'], '| TRL:', report['trl_version'], '| 精度:', report['environment']['precision'])
print('平均每步耗时（秒，去掉预热）:', report['timing']['avg_step_time_s_excl_warmup'])
print('显存峰值 allocated / reserved / 总（GB）:', report['memory']['peak_mem_allocated_gb_max_over_steps'],
      report['memory']['peak_mem_reserved_gb_max_over_steps'], report['memory']['gpu_total_memory_gb'])
print('reward:', report['reward'])
print('格式正确率:', report['format_rate'])
print('截断比例（平均）:', report['truncated_frac_mean'])
print('组内全相同比例（平均 / 最后几步）:', report['frac_groups_all_same_mean'], report['frac_groups_all_same_last_k_mean'])
print('稳定性:', report['stability'])
print('TRL 实际使用的配置:', json.dumps(report['effective_grpo_config'], ensure_ascii=False))
if report.get('dropped_config_keys'):
    print('⚠️ 被当前 TRL 忽略的参数:', report['dropped_config_keys'])

df = pd.read_csv(f'{OUTPUT_DIR}/metrics.csv')
cols = ['step', 'step_time_s', 'peak_mem_allocated_gb', 'reward_mean', 'reward_std', 'frac_groups_all_same',
        'format_rate', 'mean_completion_tokens', 'truncated_frac', 'kl', 'grad_norm']
display(df[[c for c in cols if c in df.columns]].round(4))
display(Image(f'{OUTPUT_DIR}/curves.png'))

In [ ]:
# 11. 估算正式训练的 GPU 小时（用试跑实测的每步耗时）
!python -m train.grpo.estimate --report {OUTPUT_DIR}/pilot_report.json --num_questions 10178 --epochs 1 --num_generations {NUM_GENERATIONS} --per_device_train_batch_size {PER_DEVICE_BS} --gradient_accumulation_steps {GRAD_ACCUM} --grid

# 也可以在 Python 里直接试不同方案：
from train.grpo.estimate import estimate_from_report, format_estimate
for nq, ep in [(10178, 1), (5000, 1), (10178, 2)]:
    r = estimate_from_report(f'{OUTPUT_DIR}/pilot_report.json', num_questions=nq, epochs=ep,
                             num_generations=NUM_GENERATIONS, per_device_train_batch_size=PER_DEVICE_BS,
                             gradient_accumulation_steps=GRAD_ACCUM)
    print(format_estimate(r))

In [ ]:
# 12. 复制结果到 Drive（train.grpo.grpo 结束时已复制过一次；这里再复制一次，带上完整的 train_log.txt）
import shutil
dest = Path(DRIVE_DIR) / report['run_name']
shutil.copytree(OUTPUT_DIR, dest, dirs_exist_ok=True)
print('已保存到', dest)
for p in sorted(dest.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(dest)}  ({p.stat().st_size / 1e6:.1f} MB)')